In [75]:
# dataset 
import kagglehub

# utils
import shutil
from pathlib import Path
import re
from rich import print

# eda
pd.set_option('display.max_columns', None)
import pandas as pd

# libs numericas
import numpy as np

In [2]:
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

100%|███████████████████████████████████████████████████████████████████████████████| 42.6M/42.6M [00:02<00:00, 19.0MB/s]

Extracting files...


Path to dataset files: C:\Users\arthur torres\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2


In [7]:
source = Path(r"C:\Users\arthur torres\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2")
dest = Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist")

dest.mkdir(parents=True, exist_ok=True)

for file in source.iterdir():
    shutil.copy2(file, dest / file.name)

print("Arquivos copiados!")

Arquivos copiados!


### Importando o DataFrame

In [11]:
# E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist
costumers_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_customers_dataset.csv"))
geolocation_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_geolocation_dataset.csv"))
order_items_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_order_items_dataset.csv"))
order_paymentes_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_order_payments_dataset.csv"))
order_reviews_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_order_reviews_dataset.csv"))
orders_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_orders_dataset.csv"))
products_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_products_dataset.csv"))
sellers_df = pd.read_csv(Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist\olist_sellers_dataset.csv"))


#### Importando de uma maneira mais rápida

In [34]:
source_dataset = Path(r"E:\Repos\Machine-Learning\modelos_ponta_a_ponta\BR_Ecommerce\data\olist")

dfs = {}

for file_df in source_dataset.glob("*.csv"):
    name = (
        file_df.stem
            .replace("olist_", "")
            .replace("_dataset", "")
    )
    dfs[name] = pd.read_csv(file_df)
    print(f"DataFrame criado: {dfs.keys()}")


DataFrame criado: dict_keys(['customers'])

DataFrame criado: dict_keys(['customers', 'geolocation'])

DataFrame criado: dict_keys(['customers', 'geolocation', 'orders'])

DataFrame criado: dict_keys(['customers', 'geolocation', 'orders', 'order_items'])

DataFrame criado: dict_keys(['customers', 'geolocation', 'orders', 'order_items', 'order_payments'])

DataFrame criado: dict_keys(['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 
'order_reviews'])

DataFrame criado: dict_keys(['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 
'order_reviews', 'products'])

DataFrame criado: dict_keys(['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 
'order_reviews', 'products', 'sellers'])

In [35]:
print(dfs.keys())

dict_keys(['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 'order_reviews', 'products', 
'sellers'])

In [43]:
colunas_lista = list(dfs.keys())
colunas_lista

['customers',
 'geolocation',
 'orders',
 'order_items',
 'order_payments',
 'order_reviews',
 'products',
 'sellers']

In [46]:
for coluna in colunas_lista:
    print(f"Nome da coluna: {coluna}")
    dfs[coluna].info()
    print("-"*50)

Nome da coluna: customers

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


--------------------------------------------------

Nome da coluna: geolocation

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


--------------------------------------------------

... (14 outputs left out)

--------------------------------------------------

Nome da coluna: sellers

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


--------------------------------------------------

#### Dando merge nos DataFrames

In [72]:
# Com o order_id como palavra chave
df_1 = pd.merge(dfs['orders'], dfs['order_items'], on='order_id')

df_2 = pd.merge(df_1, dfs['order_payments'], on='order_id')

df_3 = pd.merge(df_2, dfs['order_reviews'], on='order_id')

# Com cosutmers_id
df_4 = pd.merge(df_3, dfs['customers'], on='customer_id')

# Com product_id
df_5 = pd.merge(df_4, dfs['products'], on='product_id')

# Com o seller_id
df = pd.merge(df_5, dfs['sellers'], on='seller_id')

# Com o geolocation_zip_code_prefix
#df_7 = pd.merge(df_6, dfs['geolocation'], on='geolocation_zip_code_prefix')


In [76]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,1,credit_card,1,18.12,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350,maua,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,3,voucher,1,2.00,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350,maua,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,2,voucher,1,18.59,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350,maua,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,1,boleto,1,141.46,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,31570,belo horizonte,SP
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,1,credit_card,3,179.12,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,14840,guariba,SP
